# Atom Training on Google Colab

Run setup, quick test, full training, and resume workflows from this notebook.


## 1) Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Configure repo and runtime variables


In [ ]:
import os

# Required
os.environ['ATOM_REPO_URL'] = 'https://github.com/<org>/<repo>.git'

# Optional
os.environ['ATOM_BRANCH'] = 'main'
os.environ['ATOM_DRIVE_REPO'] = '/content/drive/MyDrive/dev/atom'
os.environ['ATOM_WORK_REPO'] = '/content/atom'
os.environ['ATOM_INSTALL_JAX_CUDA'] = '1'
os.environ['ATOM_JAX_VERSION'] = '0.7.2'

# Optional for private repos only:
# import getpass
# os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT (repo read): ').strip()

for key in [
    'ATOM_REPO_URL',
    'ATOM_BRANCH',
    'ATOM_DRIVE_REPO',
    'ATOM_WORK_REPO',
    'ATOM_INSTALL_JAX_CUDA',
    'ATOM_JAX_VERSION',
]:
    print(f"{key}={os.environ[key]}")
print('GITHUB_TOKEN set =', 'GITHUB_TOKEN' in os.environ)


## 3) Clone (if needed) and bootstrap session

Handles stale partial clones and gives actionable clone errors.


In [ ]:
%%bash
set -euo pipefail

WORK_REPO="${ATOM_WORK_REPO:-/content/atom}"
REPO_URL="${ATOM_REPO_URL:-}"
BRANCH="${ATOM_BRANCH:-main}"
AUTH_REPO_URL="$REPO_URL"

echo "Using WORK_REPO=$WORK_REPO"
echo "Using BRANCH=$BRANCH"

if [[ -z "$REPO_URL" ]]; then
  echo "ERROR: ATOM_REPO_URL must be set before first clone."
  exit 1
fi

if [[ "$REPO_URL" == *"<org>"* || "$REPO_URL" == *"<repo>"* ]]; then
  echo "ERROR: ATOM_REPO_URL still contains placeholders. Set a real repo URL first."
  exit 1
fi

if [[ -n "${GITHUB_TOKEN:-}" && "$REPO_URL" == https://github.com/* ]]; then
  AUTH_REPO_URL="${REPO_URL/https:\/\//https:\/\/${GITHUB_TOKEN}@}"
fi

if [[ -d "$WORK_REPO" && ! -d "$WORK_REPO/.git" ]]; then
  echo "Found stale non-git directory at $WORK_REPO. Removing it..."
  rm -rf "$WORK_REPO"
fi

if [[ ! -d "$WORK_REPO/.git" ]]; then
  echo "Cloning repository..."
  if ! git clone --branch "$BRANCH" --single-branch "$AUTH_REPO_URL" "$WORK_REPO"; then
    echo "ERROR: git clone failed."
    echo "Common causes:"
    echo "  1) Wrong ATOM_REPO_URL"
    echo "  2) Branch does not exist remotely"
    echo "  3) Private repo without token"
    echo "  4) Temporary GitHub/network issue"
    exit 1
  fi
fi

cd "$WORK_REPO"
chmod +x colab_bootstrap.sh
bash colab_bootstrap.sh


## 4) Sanity checks


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python - <<'PY'
import torch
from src.training.utils.runtime_platform import detect_runtime_platform
print('torch.cuda.is_available =', torch.cuda.is_available())
print('detected runtime platform =', detect_runtime_platform())
PY

nvidia-smi || true


## 5) Quick smoke test


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python train_progressive.py \
  --mode quick \
  --device cuda \
  --use-vmap \
  --output-dir /content/drive/MyDrive/atom_runs/quick_test


## 6) Full training run


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python train_progressive.py \
  --mode complete \
  --device cuda \
  --use-vmap \
  --timesteps 2000000 \
  --population 8 \
  --generations 10 \
  --output-dir /content/drive/MyDrive/atom_runs/run1


## 7) Resume interrupted run


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python resume_population_training.py \
  --checkpoint-dir /content/drive/MyDrive/atom_runs/run1 \
  --start-gen 8 \
  --total-gens 20 \
  --use-vmap


## Troubleshooting

- If the repo is public, do not set `GITHUB_TOKEN`.
- If clone fails, verify `ATOM_REPO_URL` and that `ATOM_BRANCH` exists remotely.
- If GPU is missing, use Runtime -> Change runtime type -> GPU.
- Save outputs/checkpoints under `/content/drive/MyDrive/...` for persistence.
